# Cell-Based, Murine, Biochemical, Histological, Transcriptomic, and Protein Data for Jasmine Leaf Extract Intervention in MASLD Models Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the [mlcroissant](https://github.com/mlcroissant/mlcroissant) library. All dataset elements are referenced by their `@id` fields to ensure reproducibility and semantic correctness.

### Dataset Source
The dataset is defined through a Croissant schema here:
* https://sen.science/doi/10.71728/senscience.aadx-dr6d/fair2.json

This notebook follows the FAIR^2 schema principles and utilizes `mlcroissant` for data loading and analysis.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.aadx-dr6d/fair2.json'

# Load the dataset metadata (ds is the Dataset object)
ds = mlc.Dataset(croissant_url)
metadata = ds.metadata

print(f"Dataset: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, columns and their respective `@id` values.

We will display all record sets in the dataset, and for each, show its fields (columns) and their `@id`.

In [ ]:
# List all record sets and their fields
record_sets = ds.record_sets

print('Record Sets in Dataset:')
for rset in record_sets:
    print(f"- {rset['@id']}  (name: {rset.get('name', '')})")
    fields = rset.get('field', [])
    # If single field, wrap in list
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        fname = field.get('name', '')
        print(f"    Field: {field['@id']} (name: {fname})")
    columns = rset.get('column', [])
    # If single column, wrap in list
    if isinstance(columns, dict):
        columns = [columns]
    for column in columns:
        cname = column.get('name', '')
        print(f"    Column: {column['@id']} (name: {cname})")


### Preview Records from a Record Set

We will preview some records from a specific record set using its `@id`. Replace `<record_set_id>` below with the actual value from the overview section above, e.g., `'cr:RecordSet_1'`.

In [ ]:
# Choose a record set to preview records
# Replace the value below with a proper record set @id
example_record_set_id = ''
if record_sets:
    example_record_set_id = record_sets[0]['@id']

print(f"Showing example records from Record Set {example_record_set_id}:")
for i, record in enumerate(ds.records(record_set=example_record_set_id)):
    if i >= 3:
        break
    print(record)

## 3. Data Extraction
Load data from each record set into DataFrames. All references are by `@id`. Use the overview to fill the record set IDs.

In [ ]:
# Extract data from each record set using their @id
record_set_ids = [rset['@id'] for rset in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    recs = list(ds.records(record_set=record_set_id))
    if recs:
        df = pd.DataFrame(recs)
        dataframes[record_set_id] = df

# Show column names for the first available DataFrame
first_set_id = next(iter(dataframes), None)
if first_set_id is not None:
    print(f"Columns in Record Set {first_set_id}:")
    print(dataframes[first_set_id].columns.tolist())
    print(dataframes[first_set_id].head())
else:
    print("No dataframes loaded; check record set availability.")

## 4. Exploratory Data Analysis (EDA)
We demonstrate filtering, normalization, and grouping using field and column `@id`. Replace `<numeric_field_id>` and `<group_field_id>` with actual IDs from the overview.

In [ ]:
# Example EDA on one record set
record_set_id = first_set_id
df = dataframes.get(record_set_id)

if df is not None:
    # Attempt to find a numeric column
    numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
    if numeric_fields:
        numeric_field_id = numeric_fields[0] # Use the first numeric column as an example
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
        print(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to find a categorical/group field
        group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
        if group_fields:
            group_field_id = group_fields[0] # Use the first group field as example
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found in the DataFrame.")
else:
    print("No DataFrame to analyze.")

## 5. Visualization
Visualize the distribution of a numeric field and relationship between fields.

Replace the field or column `@id` values accordingly. For demonstration, uses the first numeric field found.

In [ ]:
if df is not None and numeric_fields:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_fields:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
We have demonstrated loading and processing the FAIR^2 dataset using `mlcroissant`, referencing all entities by `@id`.

Key findings:
- The dataset contains multiple record sets, each with rich metadata and detailed fields.
- Data extraction is performed entirely via schema-compliant access to record sets and fields.
- Example EDA and visualization demonstrate filtering and grouping.

For further analysis, consult the full schema and documentation of each record set and field.